In [ ]:
import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer

# --- 1. АВТО-ОПРЕДЕЛЕНИЕ ЖЕЛЕЗА ---




def get_device_config():
    """
    Определяет лучшее доступное устройство и тип данных.
    RTX 5070 -> cuda, float16
    Core Ultra 5 -> cpu, float32
    """
    if torch.cuda.is_available():
        print(f"🚀 Detected GPU: {torch.cuda.get_device_name(0)}")
        return "cuda", torch.float16
    elif torch.xpu.is_available():
        print(f"🚄 Device: INTEL XPU ({torch.xpu.get_device_name(0)})")
        # Intel XPU отлично работает с float16 (bfloat16 даже лучше, но начнем с fp16)
        return "xpu", torch.float16
    else:
        # Для Core Ultra можно было бы использовать IPEX (Intel Extension), 
        # но для простоты берем стандартный CPU
        print(f"🐢 GPU not found. Using CPU (Intel Core Ultra detected/assumed).")
        return "cpu", torch.float32

DEVICE, DTYPE = get_device_config()

# --- 2. RETRIEVER (Поиск) ---
# Загружаем модель для эмбеддингов
# Она легкая, поэтому на Core Ultra тоже будет работать быстро
embedder = SentenceTransformer('all-MiniLM-L6-v2', device=DEVICE)

# Данные (AG News)
print("--- Индексация базы знаний ---")
dataset = load_dataset("ag_news", split="train")
docs = dataset['text'][:2000] # Берем 2000 для теста

# Создаем эмбеддинги
# Если DEVICE='cpu', это займет чуть больше времени, чем на RTX
doc_embeddings = embedder.encode(docs, convert_to_numpy=True, show_progress_bar=True)

# FAISS (Индекс)
d = doc_embeddings.shape[1]
index = faiss.IndexFlatL2(d)
index.add(doc_embeddings)
print(f"✅ База знаний готова ({index.ntotal} документов).")

def search(query, k=3):
    query_vec = embedder.encode([query]) # Векторизуем на том же устройстве
    distances, indices = index.search(query_vec, k)
    return [docs[idx] for idx in indices[0]]

# --- 3. GENERATOR (LLM) ---
model_name = "Qwen/Qwen2.5-1.5B-Instruct"
print(f"\n--- Загрузка LLM ({model_name}) ---")

tokenizer = AutoTokenizer.from_pretrained(model_name)

# Умная загрузка:
model = AutoModelForCausalLM.from_pretrained(
    model_name, 
    dtype=DTYPE,      # FP16 для RTX, FP32 для CPU
)

model.to(DEVICE)

if DEVICE == "xpu":
    try:
        import intel_extension_for_pytorch as ipex
        print("🔧 Применяю оптимизацию IPEX...")
        model = ipex.optimize(model, dtype=DTYPE)
    except ImportError:
        pass




In [ ]:
# Проверка размещения модели
try:
    # Берем первый параметр модели и смотрим его устройство
    param_device = next(model.parameters()).device
    print(f"📍 Модель физически находится на: {param_device}")
    
    if param_device.type == 'cpu':
        print("❌ ВНИМАНИЕ: Модель работает на процессоре!")
    elif param_device.type == 'cuda':
        print(f"✅ Модель на NVIDIA GPU: {torch.cuda.get_device_name(0)}")
    elif param_device.type == 'xpu':
        print("✅ Модель на INTEL XPU")
        
except NameError:
    print("⚠️ Модель не загружена в память.")



In [ ]:






def generate_rag_answer(prompt):
    # 1. Формируем шаблон чата
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    
    # 2. Токенизация с явным указанием возврата маски
    # .to(DEVICE) отправляет и input_ids, и attention_mask на GPU/XPU/CPU
    inputs = tokenizer([text], return_tensors="pt").to(DEVICE)

    # 3. Генерация с исправлениями
    with torch.no_grad():
        generated_ids = model.generate(
            input_ids=inputs.input_ids,
            attention_mask=inputs.attention_mask, # <--- ИСПРАВЛЕНИЕ 1: Передаем маску
            
            pad_token_id=tokenizer.eos_token_id,  # <--- ИСПРАВЛЕНИЕ 2: Явно говорим, что PAD = EOS
            
            max_new_tokens=200,
            temperature=0.7,
            do_sample=True                        # Включаем семплирование для разнообразия
        )
    
    # 4. Декодинг (обрезаем входной промпт)
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(inputs.input_ids, generated_ids)
    ]
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    
    return response



def define_context (user_query):
    # ШАГ А: Достаем факты из базы (AG News)
    found_docs = search(user_query, k=3)
    
    # ШАГ Б: Формируем контекст (Склеиваем новости в одну строку)
    context_str = ""
    for i, doc in enumerate(found_docs):
        context_str += f"Fact {i+1}: {doc}\n"

    return context_str, found_docs

def ask_rag(user_query):
    print(f"🔎 Ищу информацию по запросу: '{user_query}'...")
    
    context_str,found_docs = define_context(user_query)
    
    # ШАГ В: Создаем "Инженерный Промпт"
    # Мы строго приказываем модели смотреть ТОЛЬКО в контекст
    rag_prompt = f"""You are an analyst. Answer the question based ONLY on the provided context facts. 
If the answer is not in the context, say "I don't know based on these facts".

--- CONTEXT START ---
{context_str}
--- CONTEXT END ---

Question: {user_query}
"""

    # ШАГ Г: Отправляем это в LLM (используем вашу исправленную функцию)
    print("🤖 Генерирую ответ...")
    answer = generate_rag_answer(rag_prompt)
    
    return answer, found_docs, rag_prompt


In [ ]:
query = "Who defeated the Rockets?"

print(f"\n🗣️ Первый вопрос: {query}")
# Вызываем вашу функцию ask_rag
response, sources, rag_prompt = ask_rag(query)

print(f"\n💬 Ответ RAG:\n{response}")

# 1. Добавляем "Золотую запись" (ту самую новость) в наши документы
missing_news = "The Detroit Pistons defeated the Houston Rockets 87-79 to open the NBA season as defending champions."

print(f"➕ Добавляю знание: '{missing_news}'")

# Добавляем текст в список документов (в конец)
docs.append(missing_news) 

# 2. Создаем для нее эмбеддинг и обновляем индекс
new_embedding = embedder.encode([missing_news])
index.add(new_embedding)

print(f"✅ База знаний обновлена. Всего документов: {index.ntotal}")

# 3. Пробуем тот же вопрос снова
query = "Who defeated the Rockets?"

print(f"\n🗣️ Повторный вопрос: {query}")
# Вызываем вашу функцию ask_rag
response, sources, rag_prompt = ask_rag(query)

print(f"\n💬 Ответ RAG:\n{response}")

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# --- ИНТЕРФЕЙС ---

# 1. Виджеты
text_input = widgets.Text(
    placeholder='Задайте вопрос...',
    description='Вопрос:',
    layout=widgets.Layout(width='70%')
)
button_send = widgets.Button(
    description='Отправить',
    button_style='primary', # 'success', 'info', 'warning', 'danger' or ''
    icon='paper-plane'
)
output_area = widgets.Output()

# 2. Логика обработки
def on_button_click(b):
    user_query = text_input.value
    if not user_query: return
    
    # Очищаем поле ввода
    text_input.value = ''
    
    with output_area:
        print(f"👤 Вы: {user_query}")
        print("⏳ Думаю...")
        
        # --- ВЫЗОВ ВАШЕЙ НЕЙРОСЕТИ ---
        response, sources, full_prompt = ask_rag(user_query) # Замените на реальную функцию

        print("\n📚 НАЙДЕННЫЙ КОНТЕКСТ (Sources):")
        print("=" * 60)
        for i, doc in enumerate(sources):
            print(f"📄 [{i+1}] {doc}")
            print("=" * 60)

        prompt_out = widgets.Output()
        with prompt_out:
            print(full_prompt)
            
        # Оборачиваем его в Аккордеон
        accordion = widgets.Accordion(children=[prompt_out])
        accordion.set_title(0, '🔍 Нажмите, чтобы увидеть полный RAW PROMPT')
        accordion.selected_index = None # По умолчанию свернут
            
        display(accordion)

        
        print(f"🤖 Bot: {response}")
        print("-" * 50)

# Привязываем событие (нажатие Enter тоже можно привязать)
button_send.on_click(on_button_click)
text_input.on_submit(on_button_click)

# 3. Отображение
display(widgets.HBox([text_input, button_send]))
display(output_area)

In [ ]:
# --- HYBRID SEARCH SETUP (BM25 + VECTOR) ---
# Устанавливаем библиотеку для поиска по ключевым словам
# !pip install rank_bm25

from rank_bm25 import BM25Okapi

print("🧮 Инициализация Hybrid Search...")

# 1. Токенизация для BM25 (разбиваем текст на слова)
# В продакшене тут нужен нормальный лемматизатор, для демо хватит split()
tokenized_corpus = [doc.lower().split() for doc in docs]

# 2. Создание индекса BM25
bm25 = BM25Okapi(tokenized_corpus)

def search_hybrid(query, k=5):
    """
    Гибридный поиск: объединяет результаты BM25 (Keyword) и FAISS (Vector).
    Идеально для случаев, когда есть редкие слова или артикулы.
    """
    # А. Поиск по ключевым словам (BM25)
    query_tokens = query.lower().split()
    # Получаем top-k документов (возвращает сами тексты)
    bm25_docs = bm25.get_top_n(query_tokens, docs, n=k)
    
    # Б. Векторный поиск (FAISS)
    vector_docs = search (query, k=k)
    
    
    # В. Объединение (Union)
    # Используем set для удаления дубликатов, сохраняя порядок через list
    combined_docs = list(set(bm25_docs + vector_docs))
    
    print(f"   🔍 BM25 нашел: {len(bm25_docs)} | Vector нашел: {len(vector_docs)}")
    print(f"   ∑ После объединения: {len(combined_docs)} уникальных кандидатов")
    
    return combined_docs

In [ ]:
# --- ADVANCED RAG: 
# Устанавливаем библиотеку (если не установлена)
# !pip install sentence-transformers

from sentence_transformers import CrossEncoder

print("--- Загрузка модели ре-ранкера ---")
# Используем легкую модель MS MARCO, обученную специально определять релевантность
# Она работает как "Судья": берет пару (Вопрос, Текст) и выдает оценку от -10 до 10
reranker = CrossEncoder('BAAI/bge-reranker-base', device=DEVICE)
print("✅ Re-ranker готов.")

def define_reranked_context(user_query, initial_k, final_k):
    print(f"\n🔎 [1] Векторный поиск (Retrieve): ищу {initial_k} кандидатов...")
    
    # 1. Сначала достаем МНОГО кандидатов (Initial Retrieval)
    # Используем старую функцию search, но просим больше документов (initial_k=20)
    candidates = search_hybrid(user_query, k=initial_k)
    
    # 2. Переранжирование (Re-ranking)
    print(f"⚖️ [2] Re-ranking: оцениваю релевантность...")
    
    # Готовим пары для модели: [[Query, Doc1], [Query, Doc2], ...]
    pairs = [[user_query, doc] for doc in candidates]
    
    # Получаем оценки (scores)
    scores = reranker.predict(pairs)
    
    # Соединяем документы с их оценками и сортируем
    # scored_docs будет списком кортежей: (Text, Score)
    scored_docs = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)
    
    # 3. Вывод результатов реранкинга (для демонстрации)
    print("\n--- Результаты переранжирования ---")
    for i, (doc, score) in enumerate(scored_docs[:5]): # Показываем топ-5
        print(f"Rank {i+1} (Score: {score:.2f}): {doc[:80]}...")
        
    # 4. Отбираем лучшие для LLM
    final_docs = [doc for doc, score in scored_docs[:final_k]]
    # 5. Генерация (как раньше)
    context_str = ""
    for i, doc in enumerate(final_docs):
        context_str += f"Fact {i+1}: {doc}\n"

    return context_str, final_docs

def ask_rag_with_rerank(user_query, initial_k=20, final_k=3):
    context_str, final_docs = define_reranked_context (user_query, initial_k, final_k)      
        
    rag_prompt = f"""You are an analyst. Answer based ONLY on context facts.
--- CONTEXT START ---
{context_str}
--- CONTEXT END ---    
Question: {user_query}
"""
    print(f"\n🤖 [3] Генерация ответа LLM...")
    answer = generate_rag_answer(rag_prompt)
    return answer, final_docs, rag_prompt

In [ ]:
# --- ПОДГОТОВКА ДЕМО: ДОБАВЛЯЕМ "ШУМ" ---
tricky_query = "Why did apple fall?"

demo_docs = [
    # ПРАВИЛЬНЫЙ ОТВЕТ (Финансы. Нет слова "Apple" и "Fall", но есть "iPhone", "Stock", "Decline")
    "Market reports indicate that poor iPhone sales led to a significant stock decline.",
    
    # ЛОВУШКА 1 (Физика/Фрукты. Идеальное совпадение слов "Apple" и "Fall")
    "Isaac Newton sat under a tree and watched an apple fall to the ground.",
    
    # ЛОВУШКА 2 (Кулинария)
    "If you let the apple fall into the pie mix, it will taste better.",
    
    # ЛОВУШКА 3 (Шум)
    "Autumn leaves fall early this year."
]


for doc in demo_docs:
    docs.append(doc)
    embedding = embedder.encode([doc])
    index.add(embedding)

tokenized_corpus = [doc.lower().split() for doc in docs]
bm25 = BM25Okapi(tokenized_corpus) # Пересоздаем объект, чтобы он знал про новые документы

print(f"✅ База обновлена. Теперь в ней есть 'ловушки' для векторного поиска.")

In [ ]:
query = tricky_query

print(f"\n🗣️ Первый вопрос: {query}")
# Вызываем вашу функцию ask_rag
response, sources, rag_prompt = ask_rag(query)

print(f"\n💬 Ответ RAG:\n{response}")
print(f"\n💬 RAG prompt:\n{rag_prompt}")

In [ ]:
query = tricky_query


print(f"\n🗣️ Первый вопрос: {query}")
# Вызываем вашу функцию ask_rag
response, candidates, rag_prompt = ask_rag_with_rerank(query)

print(f"\n💬 Ответ RAG:\n{response}")
print(f"\n💬 RAG prompt:\n{rag_prompt}")